# Anomaly detection with embeddings

## Overview

This tutorial demonstrates how to use the embeddings from the Gemini API to detect potential outliers in your dataset. You will visualize a subset of the 20 Newsgroup dataset using [t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) and detect outliers outside a particular radius of the central point of each categorical cluster.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [99]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is one directory up from the notebook, hence we need to use `../` to go up one directory. If the `.env` file is in the same directory as the notebook, you can omit it altogether. 

```
│
├── .env
└── examples
    └── Anomaly_detection_with_embeddings.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [100]:
const tslab = require("tslab") as typeof import("tslab");
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


## Prepare dataset

The [20 Newsgroups Text Dataset](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset) from the open-source [SciKit project](https://scikit-learn.org/) contains 18,000 newsgroups posts on 20 topics divided into training and test sets. The split between the training and test datasets are based on messages posted before and after a specific date. This tutorial uses the training subset.


In [101]:
const fs = require("fs") as typeof import("fs");
const path = require("path") as typeof import("path");
const danfo = require("danfojs-node") as typeof import("danfojs-node");

// URL of the scikit-learn 20 Newsgroups dataset
const DATA_URL = "https://ndownloader.figshare.com/files/5975967";
const EXTRACT_PATH = "../assets/anomaly_detection";

async function downloadAndExtractDataset(): Promise<void> {
  if (fs.existsSync(EXTRACT_PATH)) {
    console.log("Dataset already exists. Skipping download.");
    return;
  }

  console.log("Downloading 20 Newsgroups dataset...");
  const response = await fetch(DATA_URL);
  const buffer = await response.arrayBuffer();

  console.log("Extracting dataset...");
  await fs.promises.mkdir(EXTRACT_PATH, { recursive: true });

  const zipPath = path.join(EXTRACT_PATH, "20news-bydate.tar.gz");
  fs.writeFileSync(zipPath, Buffer.from(buffer));

  const tar = require("tar") as typeof import("tar");
  await tar.x({
    file: zipPath,
    cwd: EXTRACT_PATH,
  });

  console.log("Dataset extracted.");
}

function loadTextFilesFromDir(dirPath: string): {
  data: string[];
  target: string[];
} {
  const categories = fs.readdirSync(dirPath);
  const data: string[] = [];
  const target: string[] = [];

  for (const category of categories) {
    const categoryPath = path.join(dirPath, category);
    if (fs.lstatSync(categoryPath).isDirectory()) {
      const files = fs.readdirSync(categoryPath);
      for (const file of files) {
        const filePath = path.join(categoryPath, file);
        const content = fs.readFileSync(filePath, "utf-8");
        data.push(content);
        target.push(category);
      }
    }
  }

  return { data, target };
}

await downloadAndExtractDataset();

const trainDir = path.join(EXTRACT_PATH, "20news-bydate-train");
const { data, target } = loadTextFilesFromDir(trainDir);

const df = new danfo.DataFrame({ data, target });

console.log("Sample Data:");
df.head().print();


Dataset already exists. Skipping download.
Sample Data:
╔════════════╤═══════════════════╤═══════════════════╗
║            │ data              │ target            ║
╟────────────┼───────────────────┼───────────────────╢
║ 0          │ From: mathew <m…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 1          │ From: mathew <m…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 2          │ From: I3150101@…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 3          │ From: mathew <m…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 4          │ From: strom@Wat…  │ alt.atheism       ║
╚════════════╧═══════════════════╧═══════════════════╝



In [102]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call */
const classNames = df.target.unique().values as string[];
console.log("Class names:", classNames);


Class names: [
  'alt.atheism',
  'comp.graphics',
  'comp.os.ms-windows.misc',
  'comp.sys.ibm.pc.hardware',
  'comp.sys.mac.hardware',
  'comp.windows.x',
  'misc.forsale',
  'rec.autos',
  'rec.motorcycles',
  'rec.sport.baseball',
  'rec.sport.hockey',
  'sci.crypt',
  'sci.electronics',
  'sci.med',
  'sci.space',
  'soc.religion.christian',
  'talk.politics.guns',
  'talk.politics.mideast',
  'talk.politics.misc',
  'talk.religion.misc'
]


Here is the first example in the training set.


In [103]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access */
const firstDoc = df.loc({ rows: [0], columns: ["data"] });
const firstText = firstDoc.data.values[0] as string;

const idx = firstText.indexOf("Lines");

if (idx !== -1) {
  console.log(firstText.slice(idx));
} else {
  console.log('"Lines" not found in the first document.');
}


Lines: 290

Archive-name: atheism/resources
Alt-atheism-archive-name: resources
Last-modified: 11 December 1992
Version: 1.0

                              Atheist Resources

                      Addresses of Atheist Organizations

                                     USA

FREEDOM FROM RELIGION FOUNDATION

Darwin fish bumper stickers and assorted other atheist paraphernalia are
available from the Freedom From Religion Foundation in the US.

Write to:  FFRF, P.O. Box 750, Madison, WI 53701.
Telephone: (608) 256-8900

EVOLUTION DESIGNS

Evolution Designs sell the "Darwin fish".  It's a fish symbol, like the ones
Christians stick on their cars, but with feet and the word "Darwin" written
inside.  The deluxe moulded 3D plastic fish is $4.95 postpaid in the US.

Write to:  Evolution Designs, 7119 Laurel Canyon #4, North Hollywood,
           CA 91605.

People in the San Francisco Bay area can get Darwin Fish from Lynn Gold --
try mailing <figmo@netcom.com>.  For net people who go to Lynn d

In [104]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment */

df.data = df.data.values.map((d: string) => {
  let cleaned = d;

  // Remove emails
  cleaned = cleaned.replace(/[\w.-]+@[\w.-]+/g, "");

  // Remove names (assuming your original regex was incomplete due to formatting)
  // You can customize this pattern based on what "names" means in your context
  cleaned = cleaned.replace(/^(.*?)(?=\n)/g, ""); // naive: remove first line, often name

  // Remove "From: "
  cleaned = cleaned.replace(/From: /g, "");

  // Remove "\nSubject: "
  cleaned = cleaned.replace(/\nSubject: /g, "");

  // Truncate to 5000 characters
  if (cleaned.length > 5000) {
    cleaned = cleaned.slice(0, 5000);
  }

  return cleaned;
});

df.head().print();


╔════════════╤═══════════════════╤═══════════════════╗
║            │ data              │ target            ║
╟────────────┼───────────────────┼───────────────────╢
║ 0          │ Alt.Atheism FAQ…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 1          │ Alt.Atheism FAQ…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: Gospel Dati…  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 3          │ Re: university …  │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────╢
║ 4          │ Re: [soc.motss,…  │ alt.atheism       ║
╚════════════╧═══════════════════╧═══════════════════╝



In [105]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment */

const texts = df.data.values as string[];
const classNameToLabelMap: Record<string, number> = df.target
  .unique()
  .values.reduce((acc: Record<string, number>, className: string, index: number) => {
    acc[className] = index + 1; // Start labels from 1
    return acc;
  }, {});

const classNames = df.target.values as string[];

const labels = classNames.map((name) => classNameToLabelMap[name]);

const df_train = new danfo.DataFrame({
  Text: texts,
  Label: labels,
  "Class Name": classNames,
});
df_train.head().print();
console.log("Training DataFrame created with", df_train.shape[0], "rows.");


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Text              │ Label             │ Class Name        ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ Alt.Atheism FAQ…  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ Alt.Atheism FAQ…  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: Gospel Dati…  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 3          │ Re: university …  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 4          │ Re: [soc.motss,…  │ 1                 │ alt.atheism       ║
╚════════════╧═══════════════════╧═══════════════════╧═══════════════════╝

Training DataFrame creat

Next, sample some of the data by taking 150 data points in the training dataset and choosing a few categories. This tutorial uses the science categories.


In [106]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument, @typescript-eslint/no-unsafe-assignment */

import { DataFrame } from "danfojs-node";

const SAMPLE_SIZE = 150;

const uniqueLabels = df_train.Label.unique().values;
const sampledGroups = [];
for (const label of uniqueLabels) {
  const labelGroup = df_train.query(df_train.Label.eq(label)).resetIndex();
  const groupSize = labelGroup.shape[0];

  if (groupSize > 0) {
    const sampledGroup = await labelGroup.sample(SAMPLE_SIZE, { seed: 42 });
    sampledGroups.push(sampledGroup);
  }
}
const df_train_sampled = danfo.concat({ dfList: sampledGroups, axis: 0 }) as DataFrame;
const df_train_final = df_train_sampled.query(df_train_sampled["Class Name"].str.includes("sci")).resetIndex();
console.log(`Sampled ${sampledGroups.length} groups from the training DataFrame.`);
console.log("Sampled DataFrame shape:", df_train_final.shape);


Sampled 20 groups from the training DataFrame.
Sampled DataFrame shape: [ 600, 3 ]


In [107]:
/* eslint-disable no-control-regex, @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument, @typescript-eslint/no-unsafe-assignment */
const cleanedText = df_train_final.Text.values.map((text: string) => text.replace(/[\x00-\x1F\x7F]/g, " "));
df_train_final.addColumn("Text", cleanedText, { inplace: true });
df_train_final.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Text              │ Label             │ Class Name        ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ Cryptography FA…  │ 12                │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ text of White H…  │ 12                │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: White House…  │ 12                │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 3          │ Cryptography FA…  │ 12                │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 4          │ Re: How do they…  │ 12                │ sci.crypt         ║
╚════════════╧═══════════════════╧═══════════════════╧═══════════════════╝



In [108]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call */

import { DataFrame } from "danfojs-node";

const valueCounts = df_train_final["Class Name"].valueCounts() as DataFrame;
valueCounts.print();


╔═════════════════╤═════╗
║ sci.crypt       │ 150 ║
╟─────────────────┼─────╢
║ sci.electronics │ 150 ║
╟─────────────────┼─────╢
║ sci.med         │ 150 ║
╟─────────────────┼─────╢
║ sci.space       │ 150 ║
╚═════════════════╧═════╝



## Create the embeddings

In this section, you will see how to generate embeddings for the different texts in the dataframe using the embeddings from the Gemini API.


### API changes to Embeddings with model embedding-001

For the embeddings model, `text-embedding-004`, there is a task type parameter and the optional title (only valid with task_type=`RETRIEVAL_DOCUMENT`).

These parameters apply only to the embeddings models. The task types are:

| Task Type             | Description                                                                  |
| --------------------- | ---------------------------------------------------------------------------- |
| `RETRIEVAL_QUERY`     | Specifies the given text is a query in a search/retrieval setting.           |
| `RETRIEVAL_DOCUMENT`  | Specifies the given text is a document in a search/retrieval setting.        |
| `SEMANTIC_SIMILARITY` | Specifies the given text will be used for Semantic Textual Similarity (STS). |
| `CLASSIFICATION`      | Specifies that the embeddings will be used for classification.               |
| `CLUSTERING`          | Specifies that the embeddings will be used for clustering.                   |


In [109]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment */

const MODEL_ID = "models/text-embedding-004";
const BATCH_SIZE = 100;
const embeddings: number[][] = [];
const display = tslab.newDisplay();
display.text("Progress: 0%");

for (let i = 0; i < df_train_final.shape[0]; i += BATCH_SIZE) {
  const batch = df_train_final.Text.values.slice(i, i + BATCH_SIZE);
  const embeddingResponse = await ai.models.embedContent({
    model: MODEL_ID,
    contents: batch,
    config: {
      taskType: "CLUSTERING",
    },
  });
  const batchEmbeddings = embeddingResponse.embeddings?.map((e) => e.values ?? []) ?? [];
  embeddings.push(...batchEmbeddings);
  display.text(`Progress: ${(((i + BATCH_SIZE) / df_train_final.shape[0]) * 100).toFixed(2)}%`);
}
df_train_final.addColumn("Embedding", new danfo.Series(embeddings), { inplace: true });


Progress: 100.00%

In [110]:
df_train_final.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Text              │ Label             │ Class Name        │ Embedding         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ Cryptography FA…  │ 12                │ sci.crypt         │ 0.024656007,0.0…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ text of White H…  │ 12                │ sci.crypt         │ 0.029945772,0.0…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: White House…  │ 12                │ sci.crypt         │ 0.00047821522,0…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 3          │ Cryptography FA…  │ 12                │ sci.crypt         │ 0.015821712,0.0…  ║
╟────────────┼───────────────────┼────────────────

## Dimensionality reduction

The dimension of the document embedding vector is 768. In order to visualize how the embedded documents are grouped together, you will need to apply dimensionality reduction as you can only visualize the embeddings in 2D or 3D space. Contextually similar documents should be closer together in space as opposed to documents that are not as similar.


In [111]:
console.log((df_train_final.at(0, "Embedding") as string).split(",").length, "dimensions in the embedding vector.");


768 dimensions in the embedding vector.


In [112]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call */
const X = df_train_final.Embedding.values.map((e: string) => e.split(",").map(Number)) as number[][];
console.log("Shape of the embedding matrix:", X.length, "x", X[0].length);


Shape of the embedding matrix: 600 x 768


You will apply the t-Distributed Stochastic Neighbor Embedding (t-SNE) approach to perform dimensionality reduction. This technique reduces the number of dimensions, while preserving clusters (points that are close together stay close together). For the original data, the model tries to construct a distribution over which other data points are "neighbors" (e.g., they share a similar meaning). It then optimizes an objective function to keep a similar distribution in the visualization.


In [113]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment */
const seedrandom = require("seedrandom") as typeof import("seedrandom");

Math.random = seedrandom("0") as () => number; // Set a fixed seed for reproducibility

const TSNE = require("tsne-js") as typeof import("tsne-js");

const tsne = new TSNE({
  dim: 2, // output dimension (2D)
  perplexity: 50, // typical value, you can tune it
  earlyExaggeration: 12.0,
  learningRate: 50.0,
  nIter: 1000, // max iterations like max_iter=1000 in sklearn
  metric: "euclidean",
});

// Initialize and run
tsne.init({
  data: X,
  type: "dense",
});
tsne.run();

// Get the 2D embeddings
const tsneResults = tsne.getOutput(); // number[][] with shape (n_samples, 2)

console.log(tsneResults);


[
  [ -2.75719412213166, -2.4167579925325673 ],
  [ -3.4650509728517473, 0.19940904059927403 ],
  [ -3.1753115790018804, -0.1284964597750167 ],
  [ -2.732233765019833, -2.2738909389327677 ],
  [ -3.640960248614705, 1.592031448202725 ],
  [ -2.913274910232553, -2.0117842994764934 ],
  [ -3.732565872724042, 0.8929089650821408 ],
  [ -2.6001813552425275, -0.4345021259512707 ],
  [ -3.4235476754065806, 0.26750584730012095 ],
  [ -2.9094321056136345, -0.20967248547410808 ],
  [ -2.670751980293048, -0.259668319473064 ],
  [ -3.0984366169794093, 0.4478399191644772 ],
  [ -3.730983535910628, 1.663066747665998 ],
  [ -2.813190116009361, -2.2528771229509585 ],
  [ 0.14338377138146707, -4.050573473374774 ],
  [ -2.7730442343509845, -1.2572076195272872 ],
  [ -2.812856057009155, -0.6776512329441536 ],
  [ -3.2991438028844975, 0.4214064649960356 ],
  [ -2.847791802459114, 0.17132417448921827 ],
  [ -3.5114410469589976, 0.09457726442168582 ],
  [ -2.481084649210149, -2.2765942963368118 ],
  [ -4.119

In [114]:
/* eslint-disable @typescript-eslint/no-unsafe-argument */
const df_tsne = new danfo.DataFrame(tsneResults, { columns: ["TSNE1", "TSNE2"] });
df_tsne.addColumn("Class Name", df_train_final["Class Name"], { inplace: true });
df_tsne.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ TSNE1             │ TSNE2             │ Class Name        ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ -2.757194122131…  │ -2.416757992532…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ -3.465050972851…  │ 0.1994090405992…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ -3.175311579001…  │ -0.128496459775…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 3          │ -2.732233765019…  │ -2.273890938932…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 4          │ -3.640960248614…  │ 1.5920314482027…  │ sci.crypt         ║
╚════════════╧═══════════════════╧═══════════════════╧═══════════════════╝



In [115]:
const rawData = df_tsne.toJSON() as { TSNE1: number[]; TSNE2: number[]; "Class Name": string[] }[];

// Step 1: Get unique class names
const classNames = [...new Set(rawData.map((row) => row["Class Name"]))];

// Step 2: Generate one trace per class
const traces = classNames.map((className) => {
  const classData = rawData.filter((row) => row["Class Name"] === className);
  return {
    x: classData.map((row) => row.TSNE1),
    y: classData.map((row) => row.TSNE2),
    mode: "markers",
    type: "scatter",
    name: className,
    marker: {
      size: 6,
    },
    text: classData.map((row) => row["Class Name"]),
    hoverinfo: "text",
  };
});

const html = `
<div style="width: 100%; height: 600px;">
  <div id="scatter-plot" style="width: 100%; height: 100%;"></div>
  <script src="https://cdn.jsdelivr.net/npm/plotly.js-dist@latest/plotly.min.js"></script>
  <script>
    const traces = ${JSON.stringify(traces)};
    
    const layout = {
      title: { text: 'Scatter plot of news using t-SNE', font: { size: 20 } },
      xaxis: { title: 'TSNE1' },
      yaxis: { title: 'TSNE2' },
      showlegend: true,
      height: 600,
      width: 800
    };

    Plotly.newPlot('scatter-plot', traces, layout);
  </script>
</div>
`;

tslab.display.html(html);


## Outlier detection

To determine which points are anomalous, you will determine which points are inliers and outliers. Start by finding the centroid, or location that represents the center of the cluster, and use the distance to determine the points that are outliers.

Start by getting the centroid of each category.


In [116]:
const centroids = df_tsne.groupby(["Class Name"]).mean();
console.log("Centroids of each class:");
centroids.print();


Centroids of each class:
╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Class Name        │ TSNE1_mean        │ TSNE2_mean        ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ sci.crypt         │ -2.957136843703…  │ -0.219018097539…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ sci.electronics   │ 0.1989684993404…  │ -1.092874851044…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ sci.med           │ 0.4772668588016…  │ 1.9774906322719…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 3          │ sci.space         │ 2.3621191261015…  │ -0.698502340616…  ║
╚════════════╧═══════════════════╧═══════════════════╧═══════════════════╝



In [117]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access */

import { DataFrame } from "danfojs-node";

function getEmbeddingCentroids(df: DataFrame): Record<string, number[]> {
  const embCentroids: Record<string, number[]> = {};
  const grouped = df.groupby(["Class Name"]);
  const uniqueClasses = Object.keys(grouped.groups);
  for (const c of uniqueClasses) {
    const subDf = grouped.getGroup([c]);
    const embeddings = (subDf.Embedding.values as string[]).map((emb) => emb.split(",").map(Number));
    const centroid = embeddings[0].map(
      (_, dim) => embeddings.reduce((sum, emb) => sum + emb[dim], 0) / embeddings.length
    );
    embCentroids[c] = centroid;
  }
  return embCentroids;
}


In [118]:
const embeddingCentroids = getEmbeddingCentroids(df_train_final);
console.log("Embedding centroids for each class:");
for (const [className, centroid] of Object.entries(embeddingCentroids)) {
  console.log(`${className}: [${centroid.slice(0, 5).join(", ")}, ...]`); // Display first 5 dimensions
}


Embedding centroids for each class:
sci.crypt: [0.022761533203466654, 0.03083421185486666, -0.04401745468546671, 0.028704286586933327, 0.022358004574199997, ...]
sci.electronics: [0.014366566139066662, 0.008163063420626665, -0.04703755094199998, 0.03413747522600001, 0.0053475741199999986, ...]
sci.med: [0.027153817067799988, 0.02117220626106667, -0.04637992124000001, 0.028582068277333336, 0.009880412332133338, ...]
sci.space: [0.05165711565620001, 0.019747807417133327, -0.03648269307839999, 0.0462917145776, 0.019814320328666667, ...]


Plot each centroid you have found against the rest of the points.


In [119]:
const centroidData = (
  danfo.toJSON(centroids) as { "Class Name": string; TSNE1_mean: number; TSNE2_mean: number }[]
).map((row) => ({
  className: row["Class Name"],
  x: row.TSNE1_mean,
  y: row.TSNE2_mean,
}));

const centroidTrace = {
  x: centroidData.map((d) => d.x),
  y: centroidData.map((d) => d.y),
  mode: "markers+text",
  type: "scatter",
  name: "Centroids",
  marker: {
    size: 14,
    color: "black",
    symbol: "x",
  },
  text: centroidData.map((d) => d.className),
  textposition: "top center",
  hoverinfo: "text",
};

const html = `
<div style="width: 100%; height: 600px;">
  <div id="scatter-plot-centroids" style="width: 100%; height: 100%;"></div>
  <script src="https://cdn.jsdelivr.net/npm/plotly.js-dist@latest/plotly.min.js"></script>
  <script>
    const allTraces = ${JSON.stringify([...traces, centroidTrace])};

    const centroidLayout = {
      title: { text: 'Scatter plot of news using t-SNE with centroids', font: { size: 20 } },
      xaxis: { title: 'TSNE1' },
      yaxis: { title: 'TSNE2' },
      showlegend: true,
      legend: { x: 1.05, y: 1 },
      height: 600,
      width: 800
    };

    Plotly.newPlot('scatter-plot-centroids', allTraces, centroidLayout);
  </script>
</div>
`;

tslab.display.html(html);


Choose a radius. Anything beyond this bound from the centroid of that category is considered an outlier.


In [120]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access */

import { DataFrame } from "danfojs-node";

function calculateEuclideanDistance(p1: number[], p2: number[]): number {
  return Math.sqrt(p1.reduce((sum, val, idx) => sum + Math.pow(val - p2[idx], 2), 0));
}

function detectOutliers(df: DataFrame, embCentroids: Record<string, number[]>, radius: number): number[] {
  const outlierFlags: boolean[] = [];
  for (let i = 0; i < df.shape[0]; i++) {
    const row = df.iloc({ rows: [i] });
    const className = row["Class Name"].values[0] as string;
    const embedding = (row.Embedding.values[0] as string).split(",").map(Number);
    const centroid = embCentroids[className];
    const distance = calculateEuclideanDistance(embedding, centroid);
    outlierFlags.push(distance > radius);
  }
  df.addColumn("Outlier", new danfo.Series(outlierFlags), { inplace: true });
  return outlierFlags.map((flag, index) => (flag ? index : -1)).filter((index) => index !== -1);
}


In [121]:
const range_ = Array.from({ length: 23 }, (_, i) => (0.3 + i * 0.02).toFixed(2));
const numOutliers: number[] = [];
for (const radius of range_) {
  const outlierCount = detectOutliers(df_train_final, embeddingCentroids, parseFloat(radius));
  numOutliers.push(outlierCount.length);
}


In [122]:
const barTrace = {
  x: range_,
  y: numOutliers,
  type: "bar",
  text: numOutliers.map(String), // bar labels
  textposition: "outside", // display on top of bars
  marker: { color: "#1f77b4" },
};

const html = `
<div style="width: 100%; height: 600px;">
  <div id="bar-plot" style="width: 100%; height: 100%;"></div>
  <script src="https://cdn.jsdelivr.net/npm/plotly.js-dist@latest/plotly.min.js"></script>
  <script>
    const barTrace = ${JSON.stringify(barTrace)};

    const barLayout = {
      title: {
        text: "Number of outliers vs. distance of points from centroid",
        font: { size: 20 }
      },
      xaxis: {
        title: { text: "Distance" },
        tickangle: -45
      },
      yaxis: {
        title: { text: "Number of outliers" }
      },
      height: 600,
      width: 1000,
      margin: { t: 80, l: 80, r: 40, b: 100 }
    };

    Plotly.newPlot("bar-plot", [barTrace], barLayout);
  </script>
</div>
`;

tslab.display.html(html);


Depending on how sensitive you want your anomaly detector to be, you can choose which radius you would like to use. For now, 0.62 is used, but you can change this value.


In [123]:
const RADIUS = 0.62;
const outlierIndices = detectOutliers(df_train_final, embeddingCentroids, RADIUS);
const df_outliers = new danfo.DataFrame(
  // @ts-expect-error false positive, expression is not callable
  df_train_final.values.filter((row, index) => outlierIndices.includes(index)),
  { columns: df_train_final.columns }
);
df_outliers.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Text              │ Label             │ Class Name        │ Embedding         │ Outlier           ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ Cryptography FA…  │ 12                │ sci.crypt         │ 0.024656007,0.0…  │ true              ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ Re: Once tapped…  │ 12                │ sci.crypt         │ 0.015565537,-0.…  │ true              ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: How do they…  │ 12                │ sci.crypt         │ 0.040472295,0.0…  │ true              ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼──────

In [124]:
const outliers_projected = df_tsne.loc({
  rows: outlierIndices,
  columns: df_tsne.columns,
});
outliers_projected.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ TSNE1             │ TSNE2             │ Class Name        ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ -2.757194122131…  │ -2.416757992532…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 7          │ -2.600181355242…  │ -0.434502125951…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 12         │ -3.730983535910…  │ 1.6630667476659…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 13         │ -2.813190116009…  │ -2.252877122950…  │ sci.crypt         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 14         │ 0.1433837713814…  │ -4.050573473374…  │ sci.crypt         ║
╚════════════╧═══════════════════╧═══════════════════╧═══════════════════╝



Plot the outliers and denote them using a transparent red color.


In [125]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-assignment */
const outlierTrace = {
  x: outliers_projected.TSNE1.values,
  y: outliers_projected.TSNE2.values,
  mode: "markers",
  type: "scatter",
  name: "Outliers",
  marker: {
    size: 10,
    color: "red",
    opacity: 0.5,
  },
  hoverinfo: "text",
};

const html = `
<div style="width: 100%; height: 600px;">
  <div id="scatter-plot-outliers" style="width: 100%; height: 100%;"></div>
  <script src="https://cdn.jsdelivr.net/npm/plotly.js-dist@latest/plotly.min.js"></script>
  <script>
    const outlierTrace = ${JSON.stringify(outlierTrace)};
    const outlierLayout = {
      title: { text: 'Scatter plot of news with outliers projected with t-SNE', font: { size: 20 } },
      xaxis: { title: 'TSNE1' },
      yaxis: { title: 'TSNE2' },
      showlegend: true,
      height: 600,
      width: 800
    };
    Plotly.newPlot('scatter-plot-outliers', [...allTraces, outlierTrace], outlierLayout);
    </script>
</div>
`;
tslab.display.html(html);


Use the index values of the datafames to print a few examples of what outliers can look like in each category. Here, the first data point from each category is printed out. Explore other points in each category to see data that are deemed as outliers, or anomalies.


In [126]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument */
const sciCryptOutliers = df_outliers.query(df_outliers["Class Name"].eq("sci.crypt"));
console.log(sciCryptOutliers.Text.values[0]);


Cryptography FAQ 07/10 - Digital Signatures Organization: The Crypt Cabal Lines: 85 Expires: 22 May 1993 04:00:07 GMT Reply-To:  NNTP-Posting-Host: pad-thai.aktis.com Summary: Part 7 of 10 of the sci.crypt FAQ, Digital Signatures and  Hash Functions.  Theory of one-way hash functions, distinctions of  terms. MD4 and MD5. Snefru. X-Last-Updated: 1993/04/16  Archive-name: cryptography-faq/part07 Last-modified: 1993/4/15   FAQ for sci.crypt, part 7: Digital Signatures and Hash Functions  This is the seventh of ten parts of the sci.crypt FAQ. The parts are mostly independent, but you should read the first part before the rest. We don't have the time to send out missing parts by mail, so don't ask. Notes such as ``[KAH67]'' refer to the reference list in the last part.  The sections of this FAQ are available via anonymous FTP to rtfm.mit.edu  as /pub/usenet/news.answers/cryptography-faq/part[xx].  The Cryptography  FAQ is posted to the newsgroups sci.crypt, sci.answers, and news.answers eve

In [127]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument */
const sciElectronicsOutliers = df_outliers.query(df_outliers["Class Name"].eq("sci.electronics"));
console.log(sciElectronicsOutliers.Text.values[0]);


Re: Conductive Plastic, what happened? Organization: Litton Systems, Toronto ONT Lines: 7  If you're thinking of reactive polymers they're making ESD safe contau iners out of it. As far as being conductive goes anything with a resistance less than 10 to the fouth  rth power ohms per cubic measure is classed as conductive per MIL-STD-1686 for ESD protection. My $0.02 ($0.016 US).  Bob. 


In [128]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument */
const sciMedOutliers = df_outliers.query(df_outliers["Class Name"].eq("sci.med"));
console.log(sciMedOutliers.Text.values[0]);


Is an oral form of Imitrex(sumatriptan) available in CA Article-I.D.: vela.1psee5$c3t Distribution: na Organization: Oakland University, Rochester MI. Lines: 9 NNTP-Posting-Host: ouchem.chem.oakland.edu  Sumatriptan(Imitrex) just became available in the US in a subcutaneous injectable form.  Is there an oral form available in CA?  A friend(yes really not me!)  has severe migranes about 2-3 times per week.  We live right by the CA border and he has gotten drugs for GERD prescribed by a US physician and filled in a CA pharmacy, but not yet FDA approved in the US.  What would be the cost of the oral form in CA$ also if anyone would have that info?      Thanks 


In [129]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument */
const sciSpaceOutliers = df_outliers.query(df_outliers["Class Name"].eq("sci.space"));
console.log(sciSpaceOutliers.Text.values[0]);


Re: pushing the envelope Article-I.D.: rave.1psogpINNksq Reply-To:  (CLAUDIO OLIVEIRA EGALON) Distribution: world Organization: NASA Langley Research Center, Hampton, VA  USA Lines: 11 NNTP-Posting-Host: tahiti.larc.nasa.gov   > flight tests are generally carefully coreographed and just what  > is going to be  'pushed' and how > far is precisely planned (despite occasional deviations from plans, > such as the 'early' first flight of the F-16 during its high-speed > taxi tests).  .. and Chuck Yeager earlier flights with the X-1...     


## Next steps

You've now created an anomaly detector using embeddings! Try using your own textual data to visualize them as embeddings, and choose some bound such that you can detect outliers. You can perform dimensionality reduction in order to complete the visualization step. Note that t-SNE is good at clustering inputs, but can take a longer time to converge or might get stuck at local minima. If you run into this issue, another technique you could consider are [principal components analysis (PCA)](https://en.wikipedia.org/wiki/Principal_component_analysis).
